In [ ]:
!CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 -q

!pip install scikit-learn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 GB 583.4 kB/s eta 0:00:00


In [ ]:

from google.colab import auth
auth.authenticate_user()

!mkdir -p /content/models
!mkdir -p /content/dataset

!gsutil cp gs://factcheck_model/models/llava-v1.6-vicuna-7b.Q4_K_M.gguf /content/models/
!gsutil cp gs://factcheck_model/models/mmproj-model-f16.gguf /content/models/

!gsutil cp gs://factcheck-bucket/dataset/train.jsonl /content/dataset/

!gsutil -m cp -r gs://factcheck-bucket/dataset/images /content/dataset/


☁️ GGUF 모델 다운로드 중...
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://factcheck_model/models/llava-v1.6-vicuna-7b.Q4_K_M.gguf...
==> NOTE: You are downloading one or more large file(s), which would
run significantly faster if you enabled sliced object downloads. This
feature is enabled by default but requires that compiled crcmod be
installed (see "gsutil help crcmod").

\ [1 files][  3.8 GiB/  3.8 GiB]   17.2 MiB/s                                   
Operation completed over 1 objects/3.8 GiB.                                      
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gclou

In [ ]:

!ls -lh /content/models/llava-v1.6-vicuna-7b.Q4_K_M.gguf
!ls -lh /content/models/mmproj-model-f16.gguf

-rw-r--r-- 1 root root 3.9G Jun  7 09:29 /content/models/llava-v1.6-vicuna-7b.Q4_K_M.gguf
-rw-r--r-- 1 root root 596M Jun  7 09:30 /content/models/mmproj-model-f16.gguf


In [ ]:
import torch
import gc
try:
    del llm
    del chat_handler
except:
    pass

gc.collect()

torch.cuda.empty_cache()


In [ ]:
import os
import json
import base64
import random
import gc
from llama_cpp import Llama
from llama_cpp.llama_chat_format import Llava16ChatHandler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


JSONL_PATH = "/content/dataset/train.jsonl"
BASE_DIR = "/content/dataset/"
MODEL_PATH = "/content/models/llava-v1.6-vicuna-7b.Q4_K_M.gguf"
PROJ_PATH = "/content/models/mmproj-model-f16.gguf"

print("🧹 메모리 정리 중...")
try:
    if 'llm' in globals(): del llm
    if 'chat_handler' in globals(): del chat_handler
except: pass
gc.collect()

all_data = []

with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        all_data.append(json.loads(line))

sample_size = min(len(all_data), 1000)
eval_data = random.sample(all_data, sample_size)

random.shuffle(eval_data)

y_true, y_pred = [], []

chat_handler = Llava16ChatHandler(clip_model_path=PROJ_PATH)
llm = Llama(
    model_path=MODEL_PATH,
    chat_handler=chat_handler,
    n_ctx=5048,
    n_gpu_layers=-1,
    n_threads=8,
    verbose=False
)

y_true, y_pred = [], []
print(f"\n🔍 총 {len(eval_data)}개 데이터 추론 시작...\n")

for i, item in enumerate(eval_data):
    img_path = os.path.join(BASE_DIR, item["image"])
    with open(img_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode('utf-8')

    human_prompt = item["conversations"][0]["value"].replace("<image>\n", "").strip()

    messages = [
        {"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}},
            {"type": "text", "text": f"{human_prompt}\n\nIs this news REAL or FAKE? Answer only with A (REAL) or B (FAKE)."}
        ]}
    ]

    response = llm.create_chat_completion(messages=messages, max_tokens=2, temperature=0.0)
    ans = response["choices"][0]["message"]["content"].strip().upper()

    pred = 1 if "B" in ans else 0
    actual = 1 if ("no image manipulation" not in item["conversations"][1]["value"].lower()) else 0

    y_true.append(actual)
    y_pred.append(pred)

    print(f"[{i+1:03d}/{len(eval_data)}] 정답: {'FAKE' if actual else 'REAL'} | 답변: {ans} -> 판별: {'FAKE' if pred else 'REAL'}")

# 결과 출력
print("\n" + "="*45)
print(f"모델 평가 결과 ({len(eval_data)}개)")
print("="*45)
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_true, y_pred, zero_division=0):.4f}")
print(f"Precision (정밀도): {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"Recall (재현율)   : {recall_score(y_true, y_pred, zero_division=0):.4f}")
print("="*45)